# Module 34 — Graph + Vector Hybrid Retrieval

Predict → Build → Try → Break → Debug → Measure → Improve → Defend

Question: does the graph signal improve retrieval enough to justify complexity?

## Benchmark contract
Hold corpus, queries, relevance labels, security policy and evaluator constant. Compare vector-only, graph-only, union/RRF and reranked strategies.

In [ ]:
from app.hybrid import hybrid, rrf
from app.benchmark import reciprocal_rank_fusion, recall_at_k, reciprocal_rank
graph=['asset-1','cve-1','control-1','team-1']
vector=['cve-1','asset-1','runbook-1','team-2']
print('BUILD hybrid=', hybrid(graph,vector,4))
print('BUILD RRF=', reciprocal_rank_fusion([graph,vector])[:4])

## TRY — retrieval metrics
Use a labeled relevance set. Retrieval quality must be measured independently from final generation quality.

In [ ]:
ranking=reciprocal_rank_fusion([graph,vector])
relevant={'cve-1','asset-1'}
print('MEASURE Recall@2=', recall_at_k(ranking,relevant,2))
print('MEASURE MRR=', reciprocal_rank(ranking,relevant))

## BREAK — graph miss
Remove the critical graph edge/candidate. Predict which multi-hop questions will fail while semantic questions may still succeed.

In [ ]:
graph_without_path=['asset-1','team-1']
print('BREAK graph-only=', graph_without_path)
print('DEBUG: compare missing path evidence against vector candidates')

## BREAK — poisoned candidate
A malicious subgraph can rank highly. Retrieval must filter tenant/ACL and validate evidence before candidate truncation and context assembly.

In [ ]:
candidates=['poisoned','cve-1','asset-1','runbook-1']
authorized={'cve-1','asset-1','runbook-1'}
safe=[x for x in candidates if x in authorized]
print('BREAK/DEFEND safe candidates=', safe)

## BREAK — traversal explosion
Increase graph fan-out/hop depth in the real lab until the traversal budget triggers. Record nodes visited and p95 latency. Never rely on an unbounded graph walk.

## Reranking experiment
Compare candidate depth 10/25/50/100. Report quality gain, p95 latency and cost. Stop increasing reranker depth when marginal task value is not worth the operational cost.

## Domain tracks
Cybersecurity: Asset → Vulnerability → Control → Owner.
Banking: Customer → Account → Transaction → Case → Policy.
Healthcare: Drug → Condition → Guideline → Evidence.
Manufacturing: Machine → Component → FailureMode → WorkOrder.
Enterprise IT: Service → Dependency → Team → Incident → Runbook.

## MEASURE scorecard
Recall@K, MRR, nDCG, multi-hop accuracy, groundedness, candidate count, traversal work, p50/p95 latency, tokens/request and cost/success.

## DEBUG challenge
A hybrid release raises Recall@20 by 7% but doubles p95 latency. Diagnose whether the gain matters for the target SLO and design the next controlled experiment.

## DEFEND / mastery gate
Build five baselines, inject retrieval/security failures, produce benchmark evidence, and write an ADR deciding whether hybrid retrieval should ship.

**Rule:** use the simplest architecture that meets quality, security, latency and cost requirements.